# Session 08 — Pandas Foundations and Exploration
## Python Fundamentals & Practical Applications — 48-Hour Edition

**STUDENT EDITION — WRITE YOUR OWN CODE**  
**Duration:** 4 classroom hours (including a 30-minute break)  
**Audience:** complete beginners and learners transferring from spreadsheets or other programming languages

> **Beginner promise:** We build one table step by step, predict results, practise each idea, and finish with a small CSV report using realistic synthetic data.
> **Working habit:** Ask a question first, inspect the data, then write the smallest useful Pandas expression.

In [116]:
student_name = input("Enter your name: ")
print(f"{student_name}, welcome to the Pandas foundations session.")

rtyyry, welcome to the Pandas foundations session.


## Learning outcomes

By the end of this session, you should be able to:

1. Explain the difference between a DataFrame and a Series.
2. Read a CSV file and inspect its shape, columns, types, and first rows.
3. Select columns and filter rows with Boolean conditions.
4. Create a calculated column.
5. Summarise categories with `value_counts()`, `describe()`, and `groupby()`.
6. Save a result as a CSV file.

**Total: 240 minutes.** Challenge work may become homework if the class needs more time.

## Google Colab quick start — ابدأ من هنا

Open the notebook in [Google Colab](https://colab.research.google.com/) or upload it using **File → Upload notebook**. Run the setup cell before the examples.

The course datasets are synthetic and are supplied in the course `datasets` folder.

In [117]:
from pathlib import Path
import sys
import pandas as pd

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    COURSE_ROOT = Path("/content/python_48h_course")
else:
    HERE = Path.cwd()
    # COURSE_ROOT = HERE if (HERE / "datasets").exists() else HERE.parent
    COURSE_ROOT = HERE

DATA_DIR = COURSE_ROOT / "datasets"
OUTPUT_DIR = COURSE_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_DATA_FILES = [
    "work_items.csv",
    "energy_readings.csv",
    "transactions_synthetic.csv",
]
missing_files = [name for name in REQUIRED_DATA_FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "Teaching datasets are missing from "
        f"{DATA_DIR}. Add the course datasets folder, then run this cell again. "
        f"Missing: {', '.join(missing_files)}"
    )

print("Dataset folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)
print("Required datasets found:", len(REQUIRED_DATA_FILES))

Dataset folder: d:\CLS_DataScience\CLS_Python\Oct 2026\3rd_edition\Python_48H_Student_Pack\Python_48H_Google_Colab_Edition\student_notebooks\datasets
Output folder: d:\CLS_DataScience\CLS_Python\Oct 2026\3rd_edition\Python_48H_Student_Pack\Python_48H_Google_Colab_Edition\student_notebooks\outputs
Required datasets found: 3


## 1. DataFrame and Series

A **DataFrame** is a table with rows and columns. A single column is usually a **Series**. A DataFrame can contain several Series that share the same row labels.

Run the example and identify which object is the table and which object is one column.

In [118]:
sample = pd.DataFrame({
    "team": ["Ahly", "Zamalek", "Pyramids"],
    "score": [82, 74, 91]
})

scores = sample["score"]
print(sample)
print("DataFrame type:", type(sample).__name__)
print("Series type:", type(scores).__name__)
print("Shape:", sample.shape)

       team  score
0      Ahly     82
1   Zamalek     74
2  Pyramids     91
DataFrame type: DataFrame
Series type: Series
Shape: (3, 2)


### Guided activity 1 — create and inspect a table

Change one score or add one row. Predict how the shape will change before running the cell.

In [119]:
# TODO: change one score or add one row, then print sample.shape


## 2. Read a CSV and inspect before analysing

The first step with an unfamiliar table is a quick inspection: size, column names, first rows, data types, and summary statistics.

`work_items.csv` contains synthetic operational records used throughout this session.

In [120]:
work = pd.read_csv(DATA_DIR / "work_items.csv")

print("Shape:", work.shape)
print("Columns:", work.columns.to_list())
print("First five rows:")
print(work.head())
print("Types:")
print(work.dtypes)

Shape: (128, 9)
Columns: ['case_id', 'sector', 'team', 'priority', 'status', 'hours_open', 'estimated_cost', 'created_date', 'owner']
First five rows:
     case_id      sector    team priority       status  hours_open  \
0  CASE-0001    Research  Team 2   medium         open        10.2   
1  CASE-0002       Audit  Team 3     high  in_progress        20.0   
2  CASE-0003    Software  Team 4      low       closed        29.8   
3  CASE-0004  Healthcare  Team 1   medium         open        39.5   
4  CASE-0005     Banking  Team 2     high  in_progress        49.2   

   estimated_cost created_date       owner  
0           137.5   2026-01-05  Learner_02  
1           175.0   2026-01-05  Learner_03  
2           212.5   2026-01-06  Learner_04  
3           248.0   2026-01-06  Learner_05  
4           285.5   2026-01-06  Learner_06  
Types:
case_id            object
sector             object
team               object
priority           object
status             object
hours_open        flo

In [121]:
# describe() summarises numeric columns
print(work[["hours_open", "estimated_cost"]].describe().round(2))

# value_counts() summarises one categorical column
print("Status counts:")
print(work["status"].value_counts())

       hours_open  estimated_cost
count      128.00          128.00
mean        30.31          899.75
std         17.27          457.74
min          0.50          120.50
25%         15.50          506.12
50%         30.15          900.75
75%         44.97         1289.12
max         59.80         1692.50
Status counts:
open           43
in_progress    43
closed         42
Name: status, dtype: int64


### Guided activity 2 — inspect a different dataset

Load `energy_readings.csv`. Print its shape, columns, and first three rows. Which column is numeric? Which column identifies a group such as a site?

In [122]:
# TODO: load energy_readings.csv and inspect it


## 3. Select columns and filter rows

Use brackets to select columns. Use `.loc[rows, columns]` when you want the row and column choices to be explicit. A Boolean condition creates `True` and `False` values; `True` rows are kept.

When combining conditions, put parentheses around each comparison.

In [123]:
selected = work[["case_id", "sector", "status", "priority"]]
print(selected.head())

is_open = work["status"] == "open"
is_medium = work["priority"] == "medium"
open_medium = work.loc[is_open & is_medium, ["case_id", "sector", "hours_open"]].copy()
print(open_medium.head())

     case_id      sector       status priority
0  CASE-0001    Research         open   medium
1  CASE-0002       Audit  in_progress     high
2  CASE-0003    Software       closed      low
3  CASE-0004  Healthcare         open   medium
4  CASE-0005     Banking  in_progress     high
      case_id      sector  hours_open
0   CASE-0001    Research        10.2
3   CASE-0004  Healthcare        39.5
6   CASE-0007          HR         8.8
9   CASE-0010       Audit        38.0
12  CASE-0013     Banking         7.2


### Selecting by label: `.loc` versus position: `.iloc`

Pandas provides two clear ways to select rows and columns:

- `.loc` uses **labels**, such as row labels and column names.
- `.iloc` uses **integer positions**, starting at zero.

Use `.loc` when the meaning of the label matters. Use `.iloc` when you deliberately want a position.

| Goal | Example |
|---|---|
| First three rows by position | `work.iloc[:3]` |
| Rows 2 through 4 by position | `work.iloc[2:5]` |
| All rows, selected columns by name | `work.loc[:, ["case_id", "sector"]]` |
| Rows matching a condition and selected columns | `work.loc[is_open, ["case_id", "status"]]` |

The end of an `.iloc` slice is excluded, as in normal Python slicing. For a label slice, `.loc` includes both endpoint labels when they exist.

In [124]:
print("First three rows and first four columns by position (.iloc):")
print(work.iloc[:3, :4])

print("First three rows and named columns by label (.loc):")
print(work.loc[:2, ["case_id", "sector", "team", "priority"]])

print("One value by position:", work.iloc[0, 1])
print("One value by labels:", work.loc[0, "sector"])

First three rows and first four columns by position (.iloc):
     case_id    sector    team priority
0  CASE-0001  Research  Team 2   medium
1  CASE-0002     Audit  Team 3     high
2  CASE-0003  Software  Team 4      low
First three rows and named columns by label (.loc):
     case_id    sector    team priority
0  CASE-0001  Research  Team 2   medium
1  CASE-0002     Audit  Team 3     high
2  CASE-0003  Software  Team 4      low
One value by position: Research
One value by labels: Research


In [125]:
# TODO: use describe() for hours_open and estimated_cost
# Then write your answers in comments.

### Guided activity 2b — practise `describe()`

Use `describe()` to summarise `hours_open` and `estimated_cost`. Then answer in a comment: which column has the larger average? What does the median tell you about a typical record?

### Guided activity 3 — build a filter in stages

First inspect `work["sector"].value_counts()`. Then choose one sector, create a Boolean condition, and print the matching case IDs. Use `.loc` to select only the columns you need.

In [126]:
# TODO: choose a sector, filter the rows, and print case_id


## 4. Create a calculated column

A calculated column turns a raw field into a useful measure. Pandas applies the operation to the whole column.

Use `.copy()` when you create a filtered table that you intend to modify, so the original table remains unchanged.

In [127]:
work["age_days"] = work["hours_open"] / 24
print(work[["case_id", "hours_open", "age_days"]].head())

     case_id  hours_open  age_days
0  CASE-0001        10.2  0.425000
1  CASE-0002        20.0  0.833333
2  CASE-0003        29.8  1.241667
3  CASE-0004        39.5  1.645833
4  CASE-0005        49.2  2.050000


## 5. Summarise categories with `groupby()`

`value_counts()` is a convenient first step when the question is simply “how many records are in each category?”.

`groupby()` is useful when we want a summary calculation for every category. We will build the idea from beginner to advanced.

### Groupby level 1 — count records

Question: how many cases belong to each sector?

The result has one value for each sector.

In [128]:
cases_by_sector = work.groupby("sector")["case_id"].count()
cases_by_sector

sector
Audit               16
Banking             16
HR                  16
Healthcare          16
Renewable Energy    16
Research            16
Software            16
Telecom             16
Name: case_id, dtype: int64

### Groupby level 2 — calculate one average

Question: what is the average number of hours open in each sector?

We select one numeric column and apply one aggregation function.

In [129]:
average_hours_by_sector = work.groupby("sector")["hours_open"].mean()
average_hours_by_sector.sort_values(ascending=False).round(2)

sector
Software            33.55
Renewable Energy    32.75
Healthcare          32.00
Audit               31.25
Telecom             29.75
Research            28.95
HR                  27.55
Banking             26.70
Name: hours_open, dtype: float64

### Groupby level 3 — calculate several summaries

Once one summary works, we can calculate a count and an average together. Named aggregations give the result useful column names.

In [142]:
sector_summary = (
    work.groupby("sector")
        .agg(
            cases=("case_id", "count"),
            average_hours=("hours_open", "mean")
        )
        .sort_values("average_hours", ascending=False)
)
sector_summary.round(1)

,cases,average_hours
sector,,
Software,16,33.6
Renewable Energy,16,32.8
Healthcare,16,32.0
Audit,16,31.2
Telecom,16,29.8
Research,16,29.0
HR,16,27.6
Banking,16,26.7


### Groupby level 4 — two grouping columns (challenge)

Question: how many cases are in each sector and priority combination?

The result has two grouping levels. `reset_index()` turns those levels back into ordinary columns, which is often easier to read and export.

In [131]:
cases_by_sector_priority = (
    work.groupby(["sector", "priority"])
        .size()
        .reset_index(name="cases")
        .sort_values("cases", ascending=False)
)
cases_by_sector_priority.head(10)

,sector,priority,cases
0,Audit,high,6
3,Banking,high,6
21,Telecom,high,6
19,Software,low,6
8,HR,medium,6
17,Research,medium,6
11,Healthcare,medium,6
13,Renewable Energy,low,6
14,Renewable Energy,medium,5
22,Telecom,low,5


### Guided activity 4 — compare groups

Create a summary by `priority` with the number of cases and average `estimated_cost`. Sort it from largest average cost to smallest.

In [132]:
# TODO: use groupby().agg() and sort_values()


## Practice tasks

Complete the guided activities first. Explain what each result means in plain English, not only what code produced it.

### Applied task A — `S08-E01`
Load `transactions_synthetic.csv`. Print its shape, column names, and the five most common transaction types.

### Applied task B — `S08-E02`
Using `work_items.csv`, select high-priority records with `.loc` and print their case IDs and estimated costs.

### Challenge — `S08-E03`
Create a summary by both `sector` and `priority`, with case count and average `hours_open`. Sort by case count descending.

In [133]:
# Write your solution for one practice task here.


## Guided mini-project — Sector performance snapshot

Create a readable CSV report from `work_items.csv`. Your report must contain one row per sector and:

- case count
- median `hours_open`
- average `estimated_cost`

Save it as `session_08_sector_snapshot.csv` in `OUTPUT_DIR`. Write one sentence explaining an observation from your report.

In [134]:
# Save the already-created sector summary as a small example CSV
example_path = OUTPUT_DIR / "example_sector_summary.csv"
sector_summary.round(2).to_csv(example_path)
print("Saved:", example_path)
print(pd.read_csv(example_path).head())

Saved: d:\CLS_DataScience\CLS_Python\Oct 2026\3rd_edition\Python_48H_Student_Pack\Python_48H_Google_Colab_Edition\student_notebooks\outputs\example_sector_summary.csv
             sector  cases  average_hours
0             Audit     16          31.25
1           Banking     16          26.70
2                HR     16          27.55
3        Healthcare     16          32.00
4  Renewable Energy     16          32.75


### Worked example — save a result as CSV

`to_csv()` writes a DataFrame to a CSV file. `index=False` prevents the Pandas row index from becoming an extra column in the file.

In [135]:
# TODO: group, aggregate, sort, and export the report
# Hint: finish with .to_csv(
#     OUTPUT_DIR / "session_08_sector_snapshot.csv",
#     index=False
# )

## Exit check — answer from memory

1. What is the difference between a DataFrame and a Series?
2. Which commands help you inspect a CSV after reading it?
3. What is the difference between `.loc` and `.iloc`?
4. How do you create a calculated column?
5. When would you use `value_counts()` instead of `groupby()`?
6. What question does `groupby()` help answer?
7. How do you save a DataFrame as a CSV?

## Common mistakes and next steps

- Inspect the column names before typing them.
- Use `==` for comparison and `=` for assignment.
- Remember that `.loc` uses labels and `.iloc` uses positions.
- Use parentheses when combining Boolean conditions.
- Do not assume the first five rows represent the whole dataset.
- Explain a grouped result in words.
- Check that an exported file exists and contains the expected columns.

**Next session:** we will clean imperfect data, handle missing values, convert dates, combine tables, and build a report.